# **Initialization**

In [1]:
"""Start"""

'Start'

In [1]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import pulp
import vrplib
import re
import sys
import os
import gc
import contextlib
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time as pytime
import pulp
from ortools.linear_solver import pywraplp
from functools import lru_cache

# --- 1. DEFINE PATH TO LIBRARY PARENT FOLDER ---
# Replace this with the ACTUAL path to the folder containing 'didp_ea_lib'
# IMPORTANT: Use r"..." string to handle Windows backslashes correctly
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm"
# --- 2. ADD TO SYSTEM PATH ---
if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
print(f"Library path added: {LIBRARY_PARENT_PATH}")
# --- 3. TEST IMPORT ---
try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import *
    from evolutionary_algorithm_lib import (compile_chromosome_to_useable_function, 
                                            combining_modified_didppy_solver_with_chromosome)
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import library. Check the path above.\nDetails: {e}")

Library path added: C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Configuration & Data input**

In [3]:
# --- CONFIGURATION ---
n_50_signal = True
if n_50_signal:
    # Path to your n50 folder containing .txt files
    DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\2_TSPTW_dual_bounds_and_models\Datasets\n50"
    INPUT_CSV = "TSPTW_single_dual_bound_50_cus_results.csv"
    OUTPUT_CSV = "result_of_ea_TSPTW_dual_bounds_50_cus.csv"
    LOGS_DIR = "TSPTW_50_cus_batch_logs"
else:
    # Path to your n20 folder containing .txt files
    DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\2_TSPTW_dual_bounds_and_models\Datasets\n20"
    INPUT_CSV = "TSPTW_single_dual_bound_20_cus_results.csv"
    OUTPUT_CSV = "result_of_ea_TSPTW_dual_bounds_20_cus.csv"
    LOGS_DIR = "TSPTW_20_cus_batch_logs"



if not os.path.exists(LOGS_DIR):
    os.makedirs(LOGS_DIR)

# --- GLOBAL VARIABLES (Initialize with Dummy Data) ---
# We create these so the functions in Cell 3 don't crash if checked early.
# These will be overwritten by the loop in Cell 4.
current_num_locations = 5
current_travel_cost = [[0.0]*5 for _ in range(5)]
current_avail_time = [0.0]*5
current_due_date = [1000.0]*5

print("✅ Globals initialized.")

# --- BATCH UTILITIES ---
def get_processed_instances(csv_path, logs_dir):
    """
    Returns a set of instances that exist in BOTH the CSV summary and the logs folder.
    """
    # 1. Get instances from CSV
    csv_instances = set()
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Instance' in df.columns:
                csv_instances = set(df['Instance'].unique())
        except:
            pass # CSV read failed, assume empty

    # 2. Get instances from Log Files
    log_instances = set()
    if os.path.exists(logs_dir):
        for filename in os.listdir(logs_dir):
            if filename.endswith("_log.txt"):
                # Extract "0.txt" from "0.txt_log.txt"
                instance_name = filename.replace("_log.txt", "")
                log_instances.add(instance_name)

    # 3. Return Intersection (Must be in BOTH to be considered "Done")
    return csv_instances.intersection(log_instances)

def append_result_to_csv(result_dict, csv_path):
    df = pd.DataFrame([result_dict])
    df.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

print("✅ Configuration set.")

def read_tsptw_data(file_path):
    """
    Reads TSPTW data from the specified file format:
    - Line 1: Number of locations (N)
    - Next N lines: Distance Matrix (N x N)
    - Next N lines: Time Windows (Ready Time, Due Date)
    - (Ignores subsequent lines, e.g., coordinates)
    """
    with open(file_path, 'r') as f:
        # Read all tokens (whitespace separated) to handle newlines flexibly
        tokens = f.read().split()
    
    iterator = iter(tokens)
    
    try:
        # 1. Number of locations
        num_locations = int(next(iterator))

        # 2. Travel Cost Matrix (N x N)
        travel_cost = []
        for _ in range(num_locations):
            row = []
            for _ in range(num_locations):
                row.append(float(next(iterator))) # Load as float
            travel_cost.append(row)
            
        # 3. Time Windows (N lines of: Ready_Time Due_Date)
        time_windows = []
        for _ in range(num_locations):
            ready = float(next(iterator)) # Load as float
            due = float(next(iterator))   # Load as float
            time_windows.append((ready, due))
        avail_time = [tw[0] for tw in time_windows]
        due_date = [tw[1] for tw in time_windows]
        return num_locations, travel_cost, avail_time, due_date

    except StopIteration:
        raise ValueError(f"Error reading file {file_path}: Unexpected end of file.")

# Cell 3.5: Data Cleanup Utility

def clean_batch_data(csv_path, logs_dir):
    """
    Ensures consistency between the CSV summary and the Log files.
    1. Removes duplicate instances in CSV (keeps last).
    2. Removes CSV rows if the corresponding Log file is missing.
    3. Deletes Log files if the corresponding CSV row is missing.
    """
    print("🧹 Starting Data Cleanup...")
    
    # 1. Load CSV
    if not os.path.exists(csv_path):
        print("   -> CSV not found. Nothing to clean in CSV.")
        # If CSV missing but logs exist, we might want to clear logs, 
        # but usually better to leave them or delete manually to be safe.
        return 

    try:
        df = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        print("   -> CSV is empty.")
        return

    original_count = len(df)
    
    # 2. Deduplicate CSV (Keep the last run)
    df.drop_duplicates(subset=['Instance'], keep='last', inplace=True)
    dedup_count = len(df)
    if original_count > dedup_count:
        print(f"   -> Removed {original_count - dedup_count} duplicate rows from CSV.")

    # 3. Remove CSV rows without matching Log files
    valid_indices = []
    instances_in_csv = set()
    
    for index, row in df.iterrows():
        instance_name = row['Instance']
        expected_log = os.path.join(logs_dir, f"{instance_name}_log.txt")
        
        if os.path.exists(expected_log):
            valid_indices.append(index)
            instances_in_csv.add(instance_name)
        else:
            print(f"   -> Removing CSV row for '{instance_name}' (Log file missing).")
            
    # Filter dataframe to keep only valid rows
    df_clean = df.loc[valid_indices]
    
    # Save cleaned CSV
    df_clean.to_csv(csv_path, index=False)
    print(f"   -> CSV saved. Current number of row is: {len(df_clean)} (was {original_count}).")

    # 4. Remove Orphan Log files (Log exists, but not in CSV)
    if os.path.exists(logs_dir):
        files = os.listdir(logs_dir)
        for filename in files:
            if filename.endswith("_log.txt"):
                instance_from_log = filename.replace("_log.txt", "")
                
                if instance_from_log not in instances_in_csv:
                    file_path = os.path.join(logs_dir, filename)
                    try:
                        os.remove(file_path)
                        print(f"   -> Deleted orphan log: {filename} (Not in CSV).")
                    except OSError as e:
                        print(f"   -> Error deleting {filename}: {e}")

    print("✨ Data Cleanup Complete.\n")


✅ Globals initialized.
✅ Configuration set.


# **Model and dual bounds declaration**

In [ ]:
def creation_of_didp_model_function():
    # Expects global variables: num_locations, dist_matrix, time_windows
    num_locations = current_num_locations
    travel_cost = current_travel_cost
    avail_time = current_avail_time
    due_date = current_due_date
    # 1. Setup Model
    model = m_dp.Model(float_cost=True)
    customer = model.add_object_type(number=num_locations)

    # 2. State Variables
    # unvisited: Set of customers to visit (excluding depot 0)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, num_locations)))
    # location: Current node
    location = model.add_element_var(object_type=customer, target=0)
    # time: Current cumulative time (resource)
    curr_time = model.add_float_resource_var(target=0.0, less_is_better=True)

    # 3. Data Tables & Helpers
    travel_time_table = model.add_float_table(travel_cost)
    
    # Separate time windows into lists for easy access

    # 4. Transitions: Visit Customer j
    for j in range(1, num_locations):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time_table[location, j] + m_dp.FloatExpr.state_cost(),
            preconditions=[
                unvisited.contains(j),
                # Feasibility check: Must arrive at j by its Due Date
                # Note: We can arrive early and wait, so we check if arrival <= due_date
                curr_time + travel_time_table[location, j] <= due_date[j]
            ],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
                # Time update: max(arrival_time, ready_time)
                # arrival_time = curr_time + travel_time
                (curr_time, m_dp.max(curr_time + travel_time_table[location, j], avail_time[j])),
            ],
        )
        model.add_transition(visit)

    # 5. Transition: Return to Depot (0)
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time_table[location, 0] + m_dp.FloatExpr.state_cost(),
        effects=[
            (location, 0),
            (curr_time, curr_time + travel_time_table[location, 0]),
        ],
        preconditions=[
            unvisited.is_empty(), 
            location != 0,
        ],
    )
    model.add_transition(return_to_depot)

    # 6. Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])

    for j in range(1, num_locations):
        model.add_state_constr(
            ~unvisited.contains(j) | (curr_time + travel_time_table[location, j] <= due_date[j])
        )

    # 8. Bundle
    metadata = {
        "num_locations": num_locations,
        "distance_matrix": travel_cost,
        "avail_time": avail_time,
        "due_date": due_date,
        "unvisited_var": unvisited,
        "location_var": location,
        "time_var": curr_time
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [ ]:
# ==========================================
# 1. Persistent 3-Index LP Relaxation
# ==========================================
def create_persistent_lp_relaxation_3_index_dual_bounds(metadata):
    """
    Creates a persistent LP model using Google OR-Tools (GLOP solver).
    Logic: 3-Index Formulation (Flow + MTZ)
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_locations']
    unvisited_set_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    dist_matrix = metadata['distance_matrix']
    
    # --- INITIALIZATION (Runs Once) ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0

    # Variables
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    u = {i: solver.NumVar(0, n_nodes, f'u_{i}') for i in range(n_nodes)}

    # Constraints (References stored for dynamic updating)
    cons_out = {} 
    cons_in = {}
    
    # Degree constraints
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'deg_out_{i}')
        for j in range(n_nodes):
            if i != j: c_out.SetCoefficient(x[(i, j)], 1)
        cons_out[i] = c_out
        
        c_in = solver.Constraint(0, 0, f'deg_in_{i}')
        for j in range(n_nodes):
            if i != j: c_in.SetCoefficient(x[(j, i)], 1)
        cons_in[i] = c_in

    # MTZ Constraints (Static)
    infinity = solver.infinity()
    for i in range(n_nodes):
        if i == 0: continue
        for j in range(n_nodes):
            if j == 0 or i == j: continue
            c_mtz = solver.Constraint(-infinity, n_nodes - 1, f'mtz_{i}_{j}')
            c_mtz.SetCoefficient(u[i], 1)
            c_mtz.SetCoefficient(u[j], -1)
            c_mtz.SetCoefficient(x[(i, j)], n_nodes)

    # Objective
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # --- CACHED WORKER ---
    @lru_cache(maxsize=10000)
    def _solve_3idx(active_tuple):
        # Tuple: (current_node, sorted_unvisited...)
        current_node = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_node)
        active_set.add(0) 
        
        max_u = len(active_set)

        # Update Bounds based on Active Set
        for i in range(n_nodes):
            if i in active_set:
                if i == current_node:
                    # Current: Out=1, In=0, u=0
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(0, 0)
                    u[i].SetBounds(0, 0)
                elif i == 0:
                    # Depot: Out=0, In=1
                    cons_out[i].SetBounds(0, 0)
                    cons_in[i].SetBounds(1, 1)
                    u[i].SetBounds(0, max_u)
                else:
                    # Intermediate: Out=1, In=1
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(1, 1)
                    u[i].SetBounds(0, max_u)
            else:
                # Inactive
                cons_out[i].SetBounds(0, 0)
                cons_in[i].SetBounds(0, 0)
                u[i].SetBounds(0, 0)

        solver.SetTimeLimit(100) # 100ms
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # --- WRAPPER ---
    def h_lp_relaxation_3_idx(state):
        unvisited = state[unvisited_set_var]
        current_node = state[location_var]
        
        if not unvisited and current_node == 0: return 0.0
        
        key = (current_node,) + tuple(sorted(list(unvisited)))
        return _solve_3idx(key)

    return h_lp_relaxation_3_idx


# ==========================================
# 2. Persistent 2-Index LP Relaxation
# ==========================================
def create_persistent_lp_relaxation_2_index_dual_bounds(metadata):
    """
    Creates a persistent 2-Index TSP Relaxation (Assignment + MTZ).
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_locations']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    # --- INITIALIZATION ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0

    infinity = solver.infinity()

    # Variables
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    u = {i: solver.NumVar(0, n_nodes, f'u_{i}') for i in range(1, n_nodes)}

    # Constraints
    cons_deg_out = {} 
    cons_deg_in = {}

    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'deg_out_{i}')
        for j in range(n_nodes):
            if i != j: c_out.SetCoefficient(x[(i, j)], 1)
        cons_deg_out[i] = c_out

        c_in = solver.Constraint(0, 0, f'deg_in_{i}')
        for j in range(n_nodes):
            if i != j: c_in.SetCoefficient(x[(j, i)], 1)
        cons_deg_in[i] = c_in

    # MTZ
    for i in range(1, n_nodes):
        for j in range(1, n_nodes):
            if i != j:
                c = solver.Constraint(-infinity, n_nodes - 1, f'mtz_{i}_{j}')
                c.SetCoefficient(u[j], 1)
                c.SetCoefficient(u[i], -1)
                c.SetCoefficient(x[(i, j)], n_nodes)

    # Objective
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # --- CACHED WORKER ---
    @lru_cache(maxsize=10000)
    def _solve_2idx(active_tuple):
        current_loc = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_loc)
        active_set.add(0)
        
        max_u = len(active_set)

        for i in range(n_nodes):
            if i in active_set:
                cons_deg_out[i].SetBounds(1, 1)
                cons_deg_in[i].SetBounds(1, 1)
                
                if i > 0:
                    if i == current_loc:
                        u[i].SetBounds(0, 0) # Anchoring
                    else:
                        u[i].SetBounds(0, max_u)
            else:
                cons_deg_out[i].SetBounds(0, 0)
                cons_deg_in[i].SetBounds(0, 0)
                if i > 0: u[i].SetBounds(0, 0)
        
        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # --- WRAPPER ---
    def h_lp_relaxation_2_idx(state):
        unvisited = state[unvisited_var]
        current_loc = state[location_var]
        
        if not unvisited and current_loc == 0: return 0.0

        key = (current_loc,) + tuple(sorted(list(unvisited)))
        return _solve_2idx(key)

    return h_lp_relaxation_2_idx


# ==========================================
# 3. Persistent TSPTW Relaxed Model (Big-M)
# ==========================================
def create_persistent_tsptw_lp_bound(metadata):
    """
    Creates a persistent TSPTW LP relaxation using GLOP.
    Logic: Flow + Time Propagation (Tight Big-M)
    """
    # --- Extract Static Data ---
    num_locations = metadata['num_locations']
    dist_matrix = metadata['distance_matrix']
    avail_time = metadata['avail_time']
    due_date = metadata['due_date']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    time_var = metadata['time_var']
    
    # Pre-calculate Big-M
    big_m = {}
    for i in range(num_locations):
        for j in range(num_locations):
            if i != j:
                val = due_date[i] + dist_matrix[i][j] - avail_time[j]
                big_m[(i, j)] = 10**6 # max(val, 0.0)

    # --- INITIALIZATION ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0
    
    infinity = solver.infinity()

    # Variables
    x = {} # Flow
    w = {} # Time
    for i in range(num_locations):
        w[i] = solver.NumVar(avail_time[i], due_date[i], f'w_{i}')
        for j in range(num_locations):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    # Constraints
    cons_flow = {}
    for i in range(num_locations):
        c = solver.Constraint(0, 0, f'flow_{i}')
        for j in range(num_locations):
            if i != j:
                c.SetCoefficient(x[(i, j)], 1)  # Out
                c.SetCoefficient(x[(j, i)], -1) # In
        cons_flow[i] = c

    # Time Constraints
    for i in range(num_locations):
        for j in range(num_locations):
            if i != j:
                M = big_m[(i, j)]
                if M > 0:
                    rhs = M - dist_matrix[i][j]
                    c = solver.Constraint(-infinity, rhs, f'time_{i}_{j}')
                    c.SetCoefficient(w[i], 1)
                    c.SetCoefficient(w[j], -1)
                    c.SetCoefficient(x[(i, j)], M)

    # Objective
    objective = solver.Objective()
    for i in range(num_locations):
        for j in range(num_locations):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # --- CACHED WORKER ---
    # TSPTW depends on Current Time, so we add it to the cache key.
    # To prevent cache misses due to tiny time differences, we round time.
    @lru_cache(maxsize=10000)
    def _solve_tsptw(input_tuple):
        # Tuple: (current_node, rounded_time, sorted_unvisited...)
        current_node = input_tuple[0]
        current_time = input_tuple[1]
        active_set = set(input_tuple[2:])
        active_set.add(current_node)
        active_set.add(0)

        # Update Model
        for i in range(num_locations):
            if i in active_set:
                # Flow Bounds
                if i == current_node:
                    # Start node: Out - In = 1
                    cons_flow[i].SetBounds(1, 1)
                    # Tighten Time Bound based on actual arrival
                    lb = max(avail_time[i], current_time)
                    ub = due_date[i]
                    if lb > ub: return float('inf') # Infeasible
                    w[i].SetBounds(lb, ub)
                elif i == 0:
                    # End node: Out - In = -1
                    cons_flow[i].SetBounds(-1, -1)
                    w[i].SetBounds(avail_time[i], due_date[i])
                else:
                    # Middle node: Out - In = 0
                    cons_flow[i].SetBounds(0, 0)
                    w[i].SetBounds(avail_time[i], due_date[i])

                # Enable relevant edges
                for j in range(num_locations):
                    if i != j:
                        if j in active_set: x[(i, j)].SetBounds(0, 1)
                        else: x[(i, j)].SetBounds(0, 0)
            else:
                # Inactive node
                cons_flow[i].SetBounds(0, 0)
                w[i].SetBounds(avail_time[i], due_date[i])
                for j in range(num_locations):
                    if i != j: x[(i, j)].SetBounds(0, 0)

        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL:
            return float(objective.Value())
        elif status == pywraplp.Solver.INFEASIBLE:
            return float('inf')
        return 0.0

    # --- WRAPPER ---
    def h_lp_tsptw(state):
        unvisited = state[unvisited_var]
        current_node = state[location_var]
        current_time = state[time_var]

        if not unvisited and current_node == 0: return 0.0

        # Round time to 2 decimal places to improve cache hits
        rounded_time = round(current_time, 2)
        
        key = (current_node, rounded_time) + tuple(sorted(list(unvisited)))
        return _solve_tsptw(key)

    return h_lp_tsptw

In [6]:
def dual_bound_expression_function(didp_bundle):
    """ Returns a dictionary of heuristic functions (bounds) bound to the model data. """
    
    model, metadata = didp_bundle
    
    # Extract metadata
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list) 
    avail_time = metadata['avail_time']
    due_date = metadata['due_date']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    num_locations = metadata['num_locations'] 
    
    # Pre-computations
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    min_outgoing_arr = np.min(masked_cost, axis=1)
    min_incoming_arr = np.min(masked_cost, axis=0)

    # --- Initialize LP Bounds ---
    h_lp_3idx = create_persistent_lp_relaxation_3_index_dual_bounds(metadata)
    h_lp_2idx = create_persistent_lp_relaxation_2_index_dual_bounds(metadata)
    h_lp_tsptw = create_persistent_tsptw_lp_bound(metadata)

    # ==========================================
    # COMBINATORIAL BOUNDS (Internal Workers)
    # ==========================================

    # --- Degree Average Bound ---
    @lru_cache(maxsize=100000)
    def _calc_degree(active_tuple):
        nodes = list(active_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)
        sum_in = np.sum(np.min(sub_mat, axis=0)[1:]) # Exclude Current
        sum_out = np.sum(np.min(sub_mat, axis=1)[:-1]) # Exclude Depot
        return float(0.5 * (sum_in + sum_out))

    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        
        # Tuple: (current, U..., depot)
        active_list = [curr] + sorted(list(U))
        if 0 not in active_list: active_list.append(0)
        return _calc_degree(tuple(active_list))

    # --- Global Min Flow ---
    @lru_cache(maxsize=100000)
    def _calc_min_flow_static(unvisited_tuple):
        val_out = sum(min_outgoing_arr[u] for u in unvisited_tuple)
        val_in = sum(min_incoming_arr[u] for u in unvisited_tuple)
        return val_out, val_in

    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        
        val_out, val_in = _calc_min_flow_static(tuple(sorted(list(U))))
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            val_in += min_incoming_arr[0]
        return float(max(val_out, val_in))

    # --- MST Bound ---
    @lru_cache(maxsize=100000)
    def _calc_mst(unvisited_tuple):
        if not unvisited_tuple: return 0.0
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    def h_mst(state):
        U = state[unvisited_var]
        return _calc_mst(tuple(sorted(list(U))))

    # --- 1-Tree Bound ---
    @lru_cache(maxsize=100000)
    def _calc_1tree(unvisited_tuple):
        subset = list(unvisited_tuple)
        depot_edges = sorted(cost_matrix[0, subset])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        if len(subset) > 1:
            sub_mat = cost_matrix[np.ix_(subset, subset)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0 
        return float(mst_val + e1 + e2)

    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_1tree(tuple(sorted(list(U))))

    # --- Assignment Bound ---
    @lru_cache(maxsize=100000)
    def _calc_assignment(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        row, col = linear_sum_assignment(assign_mat)
        return float(assign_mat[row, col].sum())

    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_assignment(tuple(sorted(list(U))))

    # --- Eigenvalue Bound ---
    @lru_cache(maxsize=100000)
    def _calc_eigen(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        N = len(nodes)
        if N < 2: return 0.0
        
        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
        
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])
        
        phi = 0.0
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals): phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
        return float(phi)

    def h_eigen(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_eigen(tuple(sorted(list(U))))

    # Return valid registry
    return automatic_creation_of_dual_bounds_registry(locals())

# Execution Line
dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())
display(dual_bound_functions_registry)

{'h_lp_3idx': <function __main__.create_persistent_lp_relaxation_3_index_dual_bounds.<locals>.h_lp_relaxation_3_idx(state)>,
 'h_lp_2idx': <function __main__.create_persistent_lp_relaxation_2_index_dual_bounds.<locals>.h_lp_relaxation_2_idx(state)>,
 'h_lp_tsptw': <function __main__.create_persistent_tsptw_lp_bound.<locals>.h_lp_tsptw(state)>,
 'h_degree_average': <function __main__.dual_bound_expression_function.<locals>.h_degree_average(state)>,
 'h_global_min_flow': <function __main__.dual_bound_expression_function.<locals>.h_global_min_flow(state)>,
 'h_mst': <function __main__.dual_bound_expression_function.<locals>.h_mst(state)>,
 'h_1tree': <function __main__.dual_bound_expression_function.<locals>.h_1tree(state)>,
 'h_assignment': <function __main__.dual_bound_expression_function.<locals>.h_assignment(state)>,
 'h_eigen': <function __main__.dual_bound_expression_function.<locals>.h_eigen(state)>}

# **Execution**

In [7]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 15        # Size of the population in each generation
GENERATIONS = 10           # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.02
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
# OPTIMAL_COST_REFERENCE= 400
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 10 #seconds

In [ ]:
# =========================================================
# 0. SETUP & HELPER FUNCTIONS
# =========================================================

# Define directories (Adjust these paths if necessary)
BASE_PATH_A = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\A"
BASE_PATH_X = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\X"
SEARCH_DIRS = [BASE_PATH_A, BASE_PATH_X]

INPUT_CSV = "CVRP_single_dual_bound_results_10s_lim.csv"
OUTPUT_CSV = "CVRP_EA_batch_results.csv"
LOGS_DIR = "CVRP_EA_batch_logs"
os.makedirs(LOGS_DIR, exist_ok=True)

def find_file_path(instance_name, search_dirs):
    """Helper to locate the .vrp file in the given folders."""
    for folder in search_dirs:
        # Check direct file
        potential_path = os.path.join(folder, instance_name)
        if os.path.exists(potential_path):
            return potential_path
        # Check recursive just in case
        files = glob.glob(os.path.join(folder, "**", instance_name), recursive=True)
        if files:
            return files[0]
    return None

def update_globals_for_cvrp(file_path):
    """Updates the specific global variables required by creation_of_didp_model_function."""
    global current_capacity, current_num_locations, current_num_vehicles, current_cust_demands, current_travel_cost, current_optimal_cost
    
    instance = vrplib.read_instance(file_path)
    
    # 1. Update Capacity & Dimensions
    current_capacity = instance['capacity']
    current_num_locations = instance['dimension']
    
    # 2. Update Num Vehicles (Regex or Fallback)
    match_trucks = re.search(r"No of trucks:\s*(\d+)", instance.get('comment', ''))
    if match_trucks:
        current_num_vehicles = int(match_trucks.group(1))
    else:
        # Fallback logic for X-series or if comment is missing
        match_filename = re.search(r"-k(\d+)", os.path.basename(file_path))
        if match_filename:
            current_num_vehicles = int(match_filename.group(1))
        else:
            current_num_vehicles = 1 # Fallback
            
    # 3. Update Demands & Costs
    current_cust_demands = instance['demand']
    current_travel_cost = instance['edge_weight']
    
    return True

def get_processed_instances(output_csv, logs_dir):
    """Returns a set of instance names that are already finished."""
    processed = set()
    if os.path.exists(output_csv):
        try:
            df = pd.read_csv(output_csv)
            if 'Instance' in df.columns:
                processed = set(df['Instance'].tolist())
        except:
            pass
    return processed

def append_result_to_csv(data_dict, output_csv):
    """Appends a single result row to the CSV."""
    file_exists = os.path.exists(output_csv)
    with open(output_csv, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=data_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(data_dict)

# =========================================================
# 1. EXECUTION CONFIGURATION
# =========================================================
# Set this to False to force Single File Mode even if CSV exists
ENABLE_BATCH_MODE = True

# If Single Mode, specify the file and an optional reference cost (if known)
SINGLE_INSTANCE_FILE = "A-n32-k5.vrp"  # Example file name
SINGLE_INSTANCE_PATH = find_file_path(SINGLE_INSTANCE_FILE, SEARCH_DIRS)
SINGLE_INSTANCE_REF_COST = 784    # Set to 0 if unknown

# =========================================================
# LOGIC: DETERMINE MODE
# =========================================================
is_batch_run = ENABLE_BATCH_MODE and os.path.exists(INPUT_CSV)

if is_batch_run:
    print(f"🚀 MODE: BATCH RUN DETECTED (Source: {INPUT_CSV})")
    
    df_input = pd.read_csv(INPUT_CSV)
    processed = get_processed_instances(OUTPUT_CSV, LOGS_DIR)
    
    if len(processed) < len(df_input):
        print(f"   -> Resuming: {len(processed)}/{len(df_input)} instances already completed.")
    else:
        print(f"   -> All {len(processed)} instances completed. No further processing needed.")
        
    # --- BATCH LOOP ---
    for index, row in df_input.iterrows():
        instance_name = row['Instance']
        
        # Check completion (Check both CSV entry and Log file existence)
        log_file_path = os.path.join(LOGS_DIR, f"{instance_name}_log.txt") 
        if instance_name in processed and os.path.exists(log_file_path):
            continue
            
        # Get optimal cost from CSV column "Best Known Cost"
        # Handle cases where it might be missing or formatted weirdly
        try:
            optimal_cost = float(row['Best Known Cost'])
        except (ValueError, KeyError):
            optimal_cost = 0.0
            
        file_path = find_file_path(instance_name, SEARCH_DIRS)
        
        if not file_path:
            print(f"\n⚠️ Skipping {instance_name}: File not found in search directories.")
            continue

        print(f"\nProcessing {instance_name} (Ref Cost: {optimal_cost})...")
        
        try:
            # A. Update Global Data (CVRP Specific)
            update_globals_for_cvrp(file_path)
            
            # Update the global reference for the model (if used elsewhere)
            current_optimal_cost = optimal_cost

            # B. Configure Params
            params = EAHyperparameters(
                # --- 1. Population ---
                population_size=POPULATION_SIZE,          
                generations=GENERATIONS,
                crossover_rate=CROSSOVER_RATE,
                mutation_rate=MUTATION_RATE,
                elitism_rate=ELITISM_RATE,        
                # --- 2. Ranges & Constraints ---
                lb_range_of_constant=LB_range_of_constant,
                ub_range_of_constant=UB_range_of_constant,
                min_chromosome_length=min_chromosome_length,    
                max_chromosome_length=max_chromosome_length,  
                # --- 3. Operator Specifics ---
                tournament_size=tournament_size,                                    
                tournament_probability=tournament_probability,                    
                mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
                homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
                subtree_crossover_probability=subtree_crossover_probability,             
                uniform_crossover_probability=uniform_crossover_probability,             
                # --- 4. Problem Specific ---
                reference_point=optimal_cost,        
                solver_time_limit=SOLVER_TIME_LIMIT,       
                # Optional: You can override available operations if needed
                available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
            )
            
            # C. Run EA (Logging to File)
            start_time = pytime.time()
            with open(log_file_path, "w", encoding="utf-8") as f:
                with contextlib.redirect_stdout(f):
                    # Create model registry functions based on new globals                 
                    best_ind = evolution_algorithm_execution(
                        didp_model_registry=creation_of_didp_model_function,
                        dual_bound_expression_function=dual_bound_expression_function,
                        params=params
                    )
                    print("\n" + "="*40 + "\nFINAL BEST INDIVIDUAL\n" + "="*40)
                    print(best_ind)          
            
            total_time = pytime.time() - start_time
            
            # D. Save Results
            fitness_val = best_ind.get('fitness', -1) if best_ind else "FAILED"
            chrom_str = str(best_ind.get('chromosome', [])) if best_ind else "FAILED"
            
            result_data = {
                "Instance": instance_name,
                "Total_Time_(s)": round(total_time, 2),
                "Best_Fitness": fitness_val,
                "Best_Chromosome": chrom_str,
                "Log_File": log_file_path
            }
            append_result_to_csv(result_data, OUTPUT_CSV)
            print(f"   ✅ Finished! Best Fit: {fitness_val} | Time: {total_time:.2f}s")
            
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            with open(log_file_path, "a") as f:
                f.write(f"\nCRITICAL ERROR: {e}")
        finally:
            gc.collect()

else:
    # =========================================================
    # SINGLE FILE EXECUTION MODE
    # =========================================================
    print(f"🚀 MODE: SINGLE RUN (File: {SINGLE_INSTANCE_PATH})")
    
    if not SINGLE_INSTANCE_PATH or not os.path.exists(SINGLE_INSTANCE_PATH):
        print(f"❌ Error: File not found at {SINGLE_INSTANCE_PATH}")
    else:
        try:
            # A. Update Global Data
            update_globals_for_cvrp(SINGLE_INSTANCE_PATH)
            
            # Use manual ref cost or fallback
            current_optimal_cost = SINGLE_INSTANCE_REF_COST

            # B. Configure Params
            params = EAHyperparameters(
                # --- 1. Population ---
                population_size=POPULATION_SIZE,          
                generations=GENERATIONS,
                crossover_rate=CROSSOVER_RATE,
                mutation_rate=MUTATION_RATE,
                elitism_rate=ELITISM_RATE,          
                # --- 2. Ranges & Constraints ---
                lb_range_of_constant=LB_range_of_constant,
                ub_range_of_constant=UB_range_of_constant,
                min_chromosome_length=min_chromosome_length,    
                max_chromosome_length=max_chromosome_length,  
                # --- 3. Operator Specifics ---
                tournament_size=tournament_size,                                    
                tournament_probability=tournament_probability,                    
                mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
                homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
                subtree_crossover_probability=subtree_crossover_probability,             
                uniform_crossover_probability=uniform_crossover_probability,             
                # --- 4. Problem Specific ---
                reference_point=SINGLE_INSTANCE_REF_COST,        
                solver_time_limit=SOLVER_TIME_LIMIT,          
                # Optional: You can override available operations if needed
                available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
            )
            
            print("   -> Starting Evolutionary Algorithm...")
            start_time = pytime.time()          
            
            # C. Run EA (Direct Output to Console)           
            best_ind = evolution_algorithm_execution(
                didp_model_registry=creation_of_didp_model_function,
                dual_bound_expression_function=dual_bound_expression_function,
                params=params
            )
            
            total_time = pytime.time() - start_time          
            
            # D. Report
            print("\n" + "="*40)
            print("🎉 SINGLE RUN COMPLETE")
            print("="*40)
            print(f"Time Taken: {total_time:.2f}s")
            print(f"Best Fitness: {best_ind.get('fitness', 'FAILED')}")
            print(f"Best Chromosome: {best_ind.get('chromosome', 'FAILED')}")    
            
        except Exception as e:
            print(f"❌ Failed: {e}")
            import traceback
            traceback.print_exc()

print("\nDONE.")